In [1]:
!pip install langchain langchain_community langchain-openai langchain_chroma
!pip install langchain-anthropic
!pip install unstructured
!python3 -m pip install konlpy

In [2]:
import os
from glob import glob
from langchain_community.document_loaders import TextLoader
from langchain_community.document_loaders import DirectoryLoader

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
import os

os.getcwd()

'/content'

In [5]:
%pwd

'/content'

In [6]:
%cd drive/MyDrive/프로젝트/예비프로젝트/

/content/drive/MyDrive/프로젝트/예비프로젝트


In [7]:
from konlpy.tag import Okt

okt = Okt()

In [8]:
def len_okt(text):
    tokens = [token for token in okt.morphs(text)]

    return len(tokens)

def okt_tokenize(text):
    return [token for token in okt.morphs(text)]

In [9]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 텍스트 분할
input_chunk_size = 300
input_chunk_overlap = 50
text_splitter = RecursiveCharacterTextSplitter(
    separators = ["\n\n", "\n", " ", ""],
    chunk_size = input_chunk_size,
    chunk_overlap = input_chunk_overlap,
    length_function = len_okt
)

In [10]:
direct = "./data"
filetype = 'txt'
print(f"디렉토리 경로'{direct}' 아래 '{filetype}'형식의 파일들을 로드합니다.\n")
loader = DirectoryLoader(direct ,glob="*."+filetype, loader_cls=TextLoader)
docs = loader.load()
print("로드 된 파일 개수 :", len(docs))

디렉토리 경로'./data' 아래 'txt'형식의 파일들을 로드합니다.

로드 된 파일 개수 : 1


In [11]:
print(f"청크사이즈{input_chunk_size}, 청크오버랩 사이즈{input_chunk_overlap} 인 text_splitter를 실행합니다.")

청크사이즈300, 청크오버랩 사이즈50 인 text_splitter를 실행합니다.


In [12]:
texts = text_splitter.split_documents(docs)

In [13]:
print("텍스트 스플릿 된 총 청크 개수: ",len(texts))

텍스트 스플릿 된 총 청크 개수:  2394


In [14]:
from langchain_community.embeddings import HuggingFaceEmbeddings

model_name = 'nlpai-lab/KURE-v1'   # sentence bert - natural language inference(: 자연어 추론)

# model_kwargs = {'device': 'cpu'} # cpu 사용하려면
model_kwargs = {'device': 'cuda'}
encode_kwargs = {'normalize_embeddings': True}
hf_embedding_model = HuggingFaceEmbeddings(
    model_name=model_name,
    model_kwargs=model_kwargs,
    encode_kwargs=encode_kwargs
)

<ipython-input-14-91a139d88198>:8: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  hf_embedding_model = HuggingFaceEmbeddings(
/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [15]:
# 임베딩 벡터 경로 지정
# 파일 형식(txt/json)_청크-오버랩_임베딩모델
embedding_directory = './'+filetype+'_'+str(input_chunk_size)+'-'+str(input_chunk_overlap)+'_'+model_name
print(f"임베딩 벡터db 경로: {embedding_directory}")

임베딩 벡터db 경로: ./txt_300-50_nlpai-lab/KURE-v1


In [16]:
from langchain_chroma import Chroma
import os
import shutil

# save_directory 가 존재한다면 해당 경로를 db로 설정
if os.path.exists(embedding_directory):
    # 기존 DB 로드
    db = Chroma(persist_directory=embedding_directory, embedding_function=hf_embedding_model)
    print("기존 Chroma DB를 로드했습니다.")
else:
    # 새 DB 생성 및 저장
    db = Chroma.from_documents(texts, hf_embedding_model, persist_directory=embedding_directory)
    print("새로운 Chroma DB를 생성하고 저장했습니다.")

기존 Chroma DB를 로드했습니다.


In [17]:
# 문서 검색기 정의
retriever = db.as_retriever(
    search_kwargs = {'k': 3}
)

In [18]:
from langchain_core.prompts import ChatPromptTemplate

# 중괄호 안에 들어있는 애들은 변수가 될 것임.
prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            '''
            You are an assistant who provides IT employment information.
            Answer the questions using the following searched context.

            Be sure to:
            - Please answer based on the data on employment information
            (technology stack, job, major tasks, preferences, required academic background, announcement date, deadline, etc.).
            - If you don't know the answer, say you don't.

            Please answer in Korean.

            #Context:
            {context}
            ''',
        ), ("human", "{question}"),
    ]
)

In [19]:
from google.colab import userdata
import os

os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')
os.environ['LANGSMITH_ENDPOINT']= userdata.get('LANGSMITH_ENDPOINT')
os.environ['LANGSMITH_PROJECT']= userdata.get('LANGSMITH_PROJECT')
os.environ['LANGSMITH_TRACING']= userdata.get('LANGSMITH_TRACING')
os.environ['LANGSMITH_API_KEY']= userdata.get('LANGSMITH_API_KEY')
os.environ['GOOGLE_API_KEY']= userdata.get('GOOGLE_API_KEY')
os.environ["HUGGING_FACE_HUB_TOKEN"] = userdata.get("HUGGING_FACE_HUB_TOKEN")
os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")

In [20]:
import getpass
import os

if not userdata.get('OPENAI_API_KEY'):
    os.environ['OPENAI_API_KEY'] = getpass.getpass('OpenAI API Key:')

from langchain_openai import ChatOpenAI
from langchain.callbacks.streaming_stdout import StreamingStdOutCallbackHandler

llm = ChatOpenAI(
    model = "gpt-4o-mini",
    temperature = 0,
    streaming = True,
    callbacks = [StreamingStdOutCallbackHandler()],
)

In [21]:
def format_docs(docs):
    return "\n\n".join(document.page_content for document in docs)

In [22]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda, RunnablePassthrough

chain = {
    # "context" : retriever,
    "context" : retriever | RunnableLambda(format_docs),
    "question" : RunnablePassthrough()
} | prompt | llm | StrOutputParser()

In [23]:
questionList = ["QA엔지니어를 뽑는 채용공고를 알려줘.",
                "나는 mongoDB를 할 수 있어. 해당 기술을 살리려면 어디에 취업 준비를 해야 할까?",
                "서버보안 관련 업무를 하려면 뭘 준비해야 해?",
                "실무경험이 부족한 대학생이 할 수 있는 준비는 뭐가 있을까.",
                "웹퍼블리셔가 되려면 뭘 준비해야해?",
                "나는 3년차 백엔드 개발자야. 이직을 고려하고 있는데 내가 지원서를 제출할 수 있을만한 곳의 채용공고를 알려줘.",
                "AI 직무에 필요한 기술 역량은 어떤 게 있어? 일단 나는 Java을 할 줄 알아.",
                "정보처리기사 자격증은 가지고 있지만 포트폴리오가 없는 경우 지원 가능한 직무는 뭐가 있어?",
                "난 신입 게임 서버 개발자야. 현재 근무하고 있는 회사에서 아직 1년차 채 되지 않았는데, 회사가 나랑 너무 안 맞아서 이직을 고려중이야.",
                "안드로이드 개발자 직군 중에 마감일자가 5월 20일 이전인 공고를 보여줘",
                "서울시내에서 출퇴근이 편한 곳에서 일하고싶어.",
                "인공지능/머신러닝 개발 직군 중에 육아휴직이 보장되는 기업이 있을까?",
                "복장규정이 빡세지 않은 곳에서 일할래. 나는 웹 풀스택 개발자로 근무한 경험이 5년 이상이야.",
                "수평적인 분위기가 좋아.",
                "프론트엔드 개발 하기 싫다… 근데 내가 할 줄 아는게 그것뿐이긴 해.",
                "3년차 백엔드 개발자인데, 회사돈으로 책 사서 공부하고싶다.",
                "요즘 IT업계에서 제일 인기있는 기술은 뭐야?",
                "인턴을 가장 많이 뽑는 기업은 어디야?",
                "1. 개발 할 때 어떤 프로그램을 사용하나요?",
                "2. 전공생이나 취업 준비생들에게 추천하는 활동은 무엇이 있을까?",
                "3. 백엔드 수행하는데 필요한 지식은 어떤 것이 있나요? 혹은 갖추고 있어야하는 지식이 있나요?",
                "4. 안드로이드 개발자의 전망에 대해서 알려줘",
                "5. 웹퍼블리셔를 수행하면서 노하우가 있을까요?",
                "6. 직무를 결정하지 못했는데 조언해줘.",
                "7. 하루 일과가 궁금합니다.",
                "8. 회사의 조직 구성이 어떻게 되나요?",
                "9. 회사 내의 복리후생이 무엇이 있나요?",
                "10. 회사 내의 특별한 사내 문화(분위기)가 있나요?",
]

In [24]:
len(questionList)

28

In [25]:
#import time
# questionList를 받으면 for 문 돌려서 질문, 대답 연속으로 받아낼 수 있도록 하기.
for question in questionList:
    response = chain.invoke(question)
    print("\n\n")

QA 엔지니어를 뽑는 채용공고는 다음과 같습니다:

1. **회사명**: 올거나이즈코리아
   - **직무**: QA Engineer (3년 이상 경력)
   - **주요 업무**: 
     - 자동화된 테스트 파이프라인 구축
     - 제품 배포를 위한 워크플로우 지원
     - 제품 주기의 모든 측면에 참여
   - **학력**: 무관
   - **경력**: 3년 이상
   - **근무지**: 서울 서초구
   - **마감일**: 2025년 05월 18일

2. **회사명**: 화이트큐브
   - **직무**: 챌린저스 QA Manager
   - **주요 업무**: 
     - 챌린저스 프로덕트 테스트
     - 테스트 문서 작성 및 수행
     - JIRA를 통한 이슈 관리
   - **학력**: 대학교 졸업 (4년 이상)
   - **경력**: 2년 이상
   - **근무지**: 경기 성남시
   - **마감일**: 2025년 05월 14일

이 외에도 QA 관련 직무가 있을 수 있으니, 추가적인 정보가 필요하시면 말씀해 주세요.


MongoDB를 활용할 수 있는 직무로는 다음과 같은 채용 공고가 있습니다:

1. **JAVA Spring 개발자**
   - **주요 기술 스택**: MongoDB, Java, Spring, REST API 개발
   - **경력 요구**: JAVA Spring 경력 11년 이상
   - **우대 사항**: MongoDB 사용 경험

2. **서비스 및 개발 PMPO**
   - **주요 업무**: B2B 및 B2C 서비스 기획 및 운영, 신규 서비스 기획
   - **우대 사항**: 핀테크, 금융 서비스 또는 이커머스 관련 PMPO 경험

이 외에도 MongoDB를 사용하는 다양한 개발 직무가 있을 수 있으니, 관련된 기술 스택을 요구하는 채용 공고를 찾아보는 것이 좋습니다.


서버보안 관련 업무를 하려면 다음과 같은 준비가 필요합니다:

1. **학력**: 대학교 졸업이 필요합니다.
2. **경력**:

In [26]:
# input[] 안에 question대신 input을 넣으면 점수 평가 실행 X
# Context 를 반환하는 RAG 결과 반환 함수
def context_answer_rag_answer(inputs: dict):
    question = inputs["question"]

    context_docs = retriever.invoke(question)
    context_text = "\n".join([doc.page_content for doc in context_docs])
    answer = chain.invoke(question)

    # 커스텀 출력 추가
    print(f"\n질문: {question}\n답변: {answer}\n\n")

    return {
        "input": question,
        "context": context_text,
        "answer": answer,
        "question": question,
    }


In [27]:
# from langchain import hub

# # 평가자 Prompt 가져오기
# llm_evaluator_prompt = hub.pull("teddynote/context-answer-evaluator")
# llm_evaluator_prompt.pretty_print()


In [28]:
prompt_text = '''
You are an expert evaluator.

Evaluate the LLM's response by examining the relationships between the following three components:
- The question asked by the user
- The answer generated by the LLM
- The given external context

You will evaluate three dimensions, each from a specific pairwise relationship:

1. **Comprehensiveness**: How well does the *answer* (B) cover the *question* (A)?
2. **Accuracy**: How well does the *question* (A) align with the *context* (C)?
3. **Context Precision**: How well does the *answer* (B) reflect or utilize the *context* (C)?

Each dimension should be scored from **1 to 5** according to the rubric below.

---

Return the score **Exactly** in the following format (one per line):
Accuracy: [Number]
Comprehensiveness: [Number]
Context Precision: [Number]

---

**Grading rubric (1–5 points)**

Accuracy (Question ↔ Context):
1 – Contradicts the context
2 – Mostly inaccurate
3 – Somewhat accurate
4 – Mostly accurate
5 – Perfectly accurate

Comprehensiveness (Answer ↔ Question):
1 – Inadequate or irrelevant
2 – Too brief, misses major points
3 – Covers some important points
4 – Covers most key aspects
5 – Completely comprehensive

Context Precision (Answer ↔ Context):
1 – Ignores or misuses context
2 – Uses context with major errors
3 – Some relevant context used
4 – Mostly contextually precise
5 – Fully aligned with context

---

# Question:
{question}

# Answer:
{answer}

# Context:
{context}
'''


In [29]:
from langchain_core.output_parsers import StrOutputParser
from langchain.chat_models import ChatOpenAI
import os
import getpass
from langchain import hub, PromptTemplate

# API 키 설정
if not os.environ.get('OPENAI_API_KEY'):
    os.environ['OPENAI_API_KEY'] = getpass.getpass('Enter your OpenAI API Key:')

# 줄바꿈 정리용 람다
fix_newlines = RunnableLambda(lambda s: s.replace("\\n", "\n"))

llm_evaluator_prompt = PromptTemplate.from_template(prompt_text)


# 평가자 생성 - OpenAI GPT-3.5 기반 평가자
custom_llm_evaluator = (
    llm_evaluator_prompt
    | ChatOpenAI(
        model="gpt-3.5-turbo",
        temperature=0,
        openai_api_key=os.environ["OPENAI_API_KEY"]
    )
    | StrOutputParser()
    | fix_newlines
)

# 답변 생성
output = context_answer_rag_answer(
    {"question": "개발 할 때 어떤 프로그램을 사용하나요?"}
)

# 점수 평가 실행
result = custom_llm_evaluator.invoke(output)
print("\n")
print(result)

<ipython-input-29-83478ea780d5>:20: LangChainDeprecationWarning: The class `ChatOpenAI` was deprecated in LangChain 0.0.10 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-openai package and should be used instead. To use it run `pip install -U :class:`~langchain-openai` and import as `from :class:`~langchain_openai import ChatOpenAI``.
  | ChatOpenAI(


개발할 때 사용하는 프로그램은 다음과 같습니다:

1. **C++** - 주로 개발에 사용되는 프로그래밍 언어입니다.
2. **Oracle** - 데이터베이스 관리 시스템으로 사용됩니다.
3. **Jira** - 이슈 관리 시스템으로, 현업과의 이슈 공유 및 커뮤니케이션에 사용됩니다.
4. **Kims** - 또 다른 이슈 관리 시스템입니다.
5. **Kotlin, Java, Spring Framework, Spring Boot, JPA, Git, Github, AWS, SQL, Linux, Docker, K8S** - 다양한 기술 스택과 툴이 사용됩니다.
6. **Typescript, SCSS, Angular, RxJS, NginX** - 프론트엔드 개발에 사용되는 기술입니다.

이 외에도 운영 모니터링을 위한 운영 프레임워크가 제공됩니다.
질문: 개발 할 때 어떤 프로그램을 사용하나요?
답변: 개발할 때 사용하는 프로그램은 다음과 같습니다:

1. **C++** - 주로 개발에 사용되는 프로그래밍 언어입니다.
2. **Oracle** - 데이터베이스 관리 시스템으로 사용됩니다.
3. **Jira** - 이슈 관리 시스템으로, 현업과의 이슈 공유 및 커뮤니케이션에 사용됩니다.
4. **Kims** - 또 다른 이슈 관리 시스템입니다.
5. **Kotlin, Java, Spring Framework, Spring Boot, JPA, Git, Github, AWS, SQL, Linux, Docker, K8S** - 다양한 기술 스택과 툴이 사용됩니다.
6. **Typescript, SCSS, Angular, RxJS, NginX** - 프론트엔드 개발에 사용되는 기술입니다.

이 외에도 운영 모니터링을 위한 운영 프레임워크가 제공됩니다.




Accuracy: 3  
Comprehensiveness: 4  
Context Precision: 4


In [30]:
# 이거 빠지면 밑에 실행 안돌아감...
def context_answer_rag_answer_eval(inputs: dict):
    question = inputs["input"]

    context_docs = retriever.invoke(question)
    context_text = "\n".join([doc.page_content for doc in context_docs])
    answer = chain.invoke(question)

    return {
        "input": question,
        "context": context_text,
        "answer": answer,
        "question": question,
    }


In [31]:
import re
from langsmith.schemas import Run, Example

# 정규식 패턴 – 줄바꿈 대응
score_pattern = re.compile(
    r"Accuracy:\s*(\d+(?:\.\d+)?)\s*"
    r"Comprehensiveness:\s*(\d+(?:\.\d+)?)\s*"
    r"Context Precision:\s*(\d+(?:\.\d+)?)",
    re.IGNORECASE
)

def parse_three_scores(text: str) -> tuple[float, float, float]:
    match = score_pattern.search(text)
    if match:
        return tuple(float(x) for x in match.groups())
    print(f"[경고] 점수 파싱 실패: {text!r}")
    return 0.0, 0.0, 0.0

def custom_evaluator(run: Run, example: Example):
    llm_answer = run.outputs.get("answer", "")
    context = run.outputs.get("context", "")
    question = example.outputs.get("question", "")

    # 평가 결과 예: "Accuracy: 7\nComprehensiveness: 10\nContext Precision: 7"
    raw = custom_llm_evaluator.invoke({
        "question": question,
        "answer": llm_answer,
        "context": context,
    })

    accuracy, comprehensiveness, context_precision = parse_three_scores(raw)

    return [
        {"key": "accuracy", "score": accuracy},
        {"key": "comprehensiveness", "score": comprehensiveness},
        {"key": "context_precision", "score": context_precision},
    ]


In [32]:
from langsmith.evaluation import evaluate

# 데이터셋 이름 설정
dataset_name = "Project_Datasets"

# 실행
experiment_results = evaluate(
    context_answer_rag_answer_eval,
    data=dataset_name,
    evaluators=[custom_evaluator],
    experiment_prefix="CUSTOM-LLM-EVAL",
    # 실험 메타데이터 지정
    metadata={
        "variant": "Custom LLM Evaluator 활용한 평가",
    },
)


View the evaluation results for experiment: 'CUSTOM-LLM-EVAL-45e5abbe' at:
https://smith.langchain.com/o/020f8f8b-3d4e-4558-9adf-75c577bbebf6/datasets/4241cf0e-511c-4a71-b121-ef60c20b2787/compare?selectedSessions=11400496-fede-41af-bb24-6c96507ff8fc




0it [00:00, ?it/s]

QA 엔지니어를 뽑는 채용공고는 다음과 같습니다:

1. **회사명**: 올거나이즈코리아
   - **직무**: QA Engineer (3년 이상 경력)
   - **주요 업무**: 
     - 자동화된 테스트 파이프라인 구축
     - 제품 배포를 위한 워크플로우 지원
     - 제품 주기의 모든 측면에 참여
   - **학력**: 무관
   - **경력**: 3년 이상
   - **근무지**: 서울 서초구
   - **마감일**: 2025년 05월 18일

2. **회사명**: 화이트큐브
   - **직무**: 챌린저스 QA Manager
   - **주요 업무**: 
     - 여러 테스트 시나리오 기반의 제품 테스트
     - 테스트 문서 작성 및 수행
     - JIRA를 통한 이슈 관리
   - **학력**: 대학교 졸업 (4년 이상)
   - **경력**: 2년 이상
   - **근무지**: 경기 성남시
   - **마감일**: 2025년 05월 14일

이 외에도 QA 관련 직무가 있을 수 있으니, 추가적인 정보가 필요하시면 말씀해 주세요.MongoDB 기술을 활용할 수 있는 취업 기회를 찾고 있다면, 다음과 같은 직무를 고려해볼 수 있습니다:

1. **소프트웨어 개발자**: MongoDB를 사용하는 프로젝트에서 데이터베이스 설계 및 관리, API 개발 등의 업무를 수행할 수 있습니다.
2. **백엔드 개발자**: MongoDB를 데이터 저장소로 활용하여 서버 사이드 로직을 개발하는 역할을 맡을 수 있습니다.
3. **데이터 엔지니어**: 데이터 파이프라인 구축 및 데이터베이스 최적화 작업을 통해 MongoDB를 활용할 수 있습니다.

특히, MongoDB 사용 경험이 우대되는 직무를 찾는 것이 좋습니다. 예를 들어, 특정 기업에서 MongoDB를 사용하는 프로젝트에 참여하거나, 관련 기술 스택을 요구하는 공고를 찾아보는 것이 유리합니다.서버보안 관련 업무를 하려면 다음과 같은 준비가 필요합니다:

1. **학력**: 대학교 졸업 이